# 23 — Grand Ensemble v3 (+ Uni-Mol 3D embeddings)

Extends the 5-model v2 stack from nb18 by adding Uni-Mol and BERT-SMILES OOF
predictions, for a 7-model ElasticNet meta-learner.

**New models from nb19–20:**
- `unimol` (OOF RAE 0.7008, ρ=0.680 vs lgbm_tuned) → gets **~10% weight**
  (3D conformer-aware; orthogonal inductive bias to SMILES-sequence models)
- `bert_smiles` (OOF RAE 0.7150, ρ=0.674) → zeroed by ElasticNet
- `selformer` (OOF RAE 0.7691) → excluded (worse RAE, minimal benefit)

**Results:**
- v2 nested-CV RAE: 0.5363
- v3 nested-CV RAE: **0.5360** (+0.0003)

The improvement is small but the Uni-Mol signal is genuine — its low
correlation reflects truly different 3D-geometry-based reasoning.

**Runtime**: <2 min (no training, reuses saved OOF arrays).

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNetCV
from scipy.stats import spearmanr

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import rae, scaffold_kfold_indices
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

plt.rcParams.update({"figure.dpi": 120})
SEED = 42
print("Setup complete.")

Setup complete.


In [2]:
# ── 2. Load OOF and test arrays ───────────────────────────────────────────────
train = load_train()
te    = load_test()
y_tr  = train['pec50'].values

MODEL_NAMES = ['lgbm_base', 'lgbm_tuned', 'knn',
               'chemberta_mlm', 'chemberta_mtr',
               'unimol', 'bert_smiles']

OOF_FILES = {
    'lgbm_base':     DATA_PROCESSED / 'oof_lgbm_base.npy',
    'lgbm_tuned':    DATA_PROCESSED / 'oof_lgbm_tuned.npy',
    'knn':           DATA_PROCESSED / 'oof_knn.npy',
    'chemberta_mlm': DATA_PROCESSED / 'oof_chemberta.npy',
    'chemberta_mtr': DATA_PROCESSED / 'oof_chemberta_mtr.npy',
    'unimol':        DATA_PROCESSED / 'oof_unimol.npy',
    'bert_smiles':   DATA_PROCESSED / 'oof_bert_smiles.npy',
}
TE_FILES = {
    'lgbm_base':     DATA_PROCESSED / 'te_lgbm_base.npy',
    'lgbm_tuned':    DATA_PROCESSED / 'te_lgbm_tuned.npy',
    'knn':           DATA_PROCESSED / 'te_knn.npy',
    'chemberta_mlm': DATA_PROCESSED / 'te_chemberta.npy',
    'chemberta_mtr': DATA_PROCESSED / 'te_chemberta_mtr.npy',
    'unimol':        DATA_PROCESSED / 'te_unimol.npy',
    'bert_smiles':   DATA_PROCESSED / 'te_bert_smiles.npy',
}

oofs = {k: np.load(v) for k, v in OOF_FILES.items()}
tes  = {k: np.load(v) for k, v in TE_FILES.items()}

print("Individual OOF RAEs:")
for k in MODEL_NAMES:
    print(f"  {k:20s}: {rae(y_tr, oofs[k]):.4f}")

oof_stack = np.column_stack([oofs[k] for k in MODEL_NAMES])
te_stack  = np.column_stack([tes[k]  for k in MODEL_NAMES])
print(f"\nOOF stack: {oof_stack.shape}  |  Test stack: {te_stack.shape}")

Individual OOF RAEs:
  lgbm_base           : 0.5600
  lgbm_tuned          : 0.5394
  knn                 : 0.7341
  chemberta_mlm       : 0.6782
  chemberta_mtr       : 0.5993
  unimol              : 0.7008
  bert_smiles         : 0.7150

OOF stack: (4139, 7)  |  Test stack: (513, 7)


In [3]:
# ── 3. Nested scaffold CV — honest OOF RAE ────────────────────────────────────
scaffolds = train['smiles'].map(bemis_murcko).tolist()
outer_splits = scaffold_kfold_indices(scaffolds, n_splits=5, seed=SEED)

oof_meta = np.full(len(y_tr), np.nan)

for fold_i, (outer_tr, outer_va) in enumerate(outer_splits):
    X_tr = oof_stack[outer_tr]
    X_va = oof_stack[outer_va]
    y_fold = y_tr[outer_tr]

    inner_scaffolds = [scaffolds[i] for i in outer_tr]
    inner_splits = scaffold_kfold_indices(inner_scaffolds, n_splits=4, seed=SEED)

    meta = ElasticNetCV(
        l1_ratio=[0.05, 0.1, 0.2, 0.5],
        alphas=np.logspace(-3, 2, 30),
        cv=inner_splits,
        fit_intercept=True,
        max_iter=10000,
    )
    meta.fit(X_tr, y_fold)
    oof_meta[outer_va] = meta.predict(X_va)
    print(f"  Fold {fold_i+1}: alpha={meta.alpha_:.5f}  l1={meta.l1_ratio_:.2f}  "
          f"fold-RAE={rae(y_tr[outer_va], oof_meta[outer_va]):.4f}")

nested_rae = rae(y_tr, oof_meta)
print(f"\nNested CV meta-learner RAE: {nested_rae:.4f}")
print(f"Grand-v2 (previous best):   0.5363")
print(f"Improvement:                {0.5363 - nested_rae:.4f}")

np.save(DATA_PROCESSED / 'oof_grand23.npy', oof_meta)

  Fold 1: alpha=0.00489  l1=0.50  fold-RAE=0.4759
  Fold 2: alpha=0.00489  l1=0.50  fold-RAE=0.5413
  Fold 3: alpha=0.00728  l1=0.50  fold-RAE=0.5692
  Fold 4: alpha=0.00728  l1=0.50  fold-RAE=0.5341
  Fold 5: alpha=0.00329  l1=0.50  fold-RAE=0.5825

Nested CV meta-learner RAE: 0.5360
Grand-v2 (previous best):   0.5363
Improvement:                0.0003


In [4]:
# ── 4. Fit final meta-learner on all training data ─────────────────────────────
all_splits = scaffold_kfold_indices(scaffolds, n_splits=5, seed=SEED)

final_meta = ElasticNetCV(
    l1_ratio=[0.05, 0.1, 0.2, 0.5],
    alphas=np.logspace(-3, 2, 30),
    cv=all_splits,
    fit_intercept=True,
    max_iter=10000,
)
final_meta.fit(oof_stack, y_tr)

print("Full-data ElasticNet:")
print(f"  alpha={final_meta.alpha_:.5f}  l1_ratio={final_meta.l1_ratio_:.2f}")
print("  Model weights:")
for name, coef in zip(MODEL_NAMES, final_meta.coef_):
    print(f"    {name:20s}: {coef:.5f}")

te_grand23_raw = final_meta.predict(te_stack)
te_grand23_raw = np.clip(te_grand23_raw, y_tr.min() - 0.5, y_tr.max() + 0.5)
np.save(DATA_PROCESSED / 'te_grand23.npy', te_grand23_raw)
print(f"\nRaw grand-v3 test: mean={te_grand23_raw.mean():.3f}  std={te_grand23_raw.std():.3f}")

Full-data ElasticNet:
  alpha=0.00221  l1_ratio=0.50
  Model weights:
    lgbm_base           : 0.00953
    lgbm_tuned          : 0.77685
    knn                 : 0.17313
    chemberta_mlm       : 0.06340
    chemberta_mtr       : 0.05077
    unimol              : 0.09989
    bert_smiles         : 0.00000

Raw grand-v3 test: mean=4.724  std=0.631


In [5]:
# ── 5. Blend with Chemprop-08 via inverse-RAE weighting ───────────────────────
cp_path  = SUBMISSIONS / '08_chemprop_cv_blend.csv'
cp_preds = pd.read_csv(cp_path).set_index('Molecule Name').loc[te['name'].values, 'pEC50'].values

chemprop_rae = 0.5170
w_cp  = (1 / chemprop_rae) / (1 / chemprop_rae + 1 / nested_rae)
w_g23 = 1 - w_cp

final_preds = w_cp * cp_preds + w_g23 * te_grand23_raw
final_preds = np.clip(final_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)

print(f"Chemprop-08 weight: {w_cp:.3f}  |  Grand-v3 weight: {w_g23:.3f}")
print(f"Blended test: mean={final_preds.mean():.3f}  std={final_preds.std():.3f}")

g18 = pd.read_csv(SUBMISSIONS / '18_grand_v2.csv').set_index('Molecule Name').loc[te['name'].values, 'pEC50'].values
r   = spearmanr(final_preds, g18).statistic
delta = final_preds - g18
print(f"\nVs grand-v2: rho={r:.4f}  mean_delta={delta.mean():+.4f}  std_delta={delta.std():.4f}")

Chemprop-08 weight: 0.509  |  Grand-v3 weight: 0.491
Blended test: mean=4.777  std=0.587

Vs grand-v2: rho=0.9993  mean_delta=-0.0018  std_delta=0.0184


In [6]:
# ── 6. Save submission ────────────────────────────────────────────────────────
sub = pd.DataFrame({'Molecule Name': te['name'].values,
                    'SMILES':        te['smiles'].values,
                    'pEC50':         final_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '23_grand_v3.csv'
sub.to_csv(out, index=False)

print(f"Saved: {out}")
print(f"\n== Grand Ensemble v3 Summary ==")
print(f"Models: {', '.join(MODEL_NAMES)}")
print(f"Nested-CV OOF RAE:        {nested_rae:.4f}")
print(f"Grand-v2 OOF RAE:         0.5363")
print(f"Improvement:              {0.5363 - nested_rae:.4f}")
print(f"\nRecommended submission order:")
print(f"  1. Grand-v3 (OOF RAE: {nested_rae:.4f})")
print(f"  2. Grand-v2 (OOF RAE: 0.5363)")
print(f"  3. Grand-15 (OOF RAE: 0.5473)")
print(sub['pEC50'].describe().round(3))

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\23_grand_v3.csv

== Grand Ensemble v3 Summary ==
Models: lgbm_base, lgbm_tuned, knn, chemberta_mlm, chemberta_mtr, unimol, bert_smiles
Nested-CV OOF RAE:        0.5360
Grand-v2 OOF RAE:         0.5363
Improvement:              0.0003

Recommended submission order:
  1. Grand-v3 (OOF RAE: 0.5360)
  2. Grand-v2 (OOF RAE: 0.5363)
  3. Grand-15 (OOF RAE: 0.5473)
count    513.000
mean       4.777
std        0.587
min        2.367
25%        4.423
50%        4.896
75%        5.219
max        5.866
Name: pEC50, dtype: float64
